In [ ]:
import cv2 



In [2]:
import sys
print(sys.executable)

/home/joel/miniconda3/envs/aitopics/bin/python


In [3]:
import mediapipe as mp
import cv2 

print(mp.__version__)
print(hasattr(mp, "tasks"))

0.10.35
True


In [7]:
import cv2

# video_path = "your_video.mp4"

video_path = "/home/joel/Documents/music_lesson_app/backend-fastapi/video/waltz_no2_practice.mp4" 



cap = cv2.VideoCapture(video_path)

print("Opened:", cap.isOpened())
print("Frame count:", cap.get(cv2.CAP_PROP_FRAME_COUNT))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))

ret, frame = cap.read()

print("Read success:", ret)
print("Frame is None:", frame is None)

cap.release()

Opened: True
Frame count: 141.0
FPS: 30.0
Read success: True
Frame is None: False


In [12]:
import cv2
import mediapipe as mp
import numpy as np

def format_time(seconds):
    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{minutes}:{secs:02d}"

def get_head_down_score(landmarks):
    nose = landmarks[0]
    left_eye = landmarks[2]
    right_eye = landmarks[5]
    left_shoulder = landmarks[11]
    right_shoulder = landmarks[12]
    left_hip = landmarks[23]
    right_hip = landmarks[24]

    eye_y = (left_eye.y + right_eye.y) / 2
    shoulder_y = (left_shoulder.y + right_shoulder.y) / 2
    hip_y = (left_hip.y + right_hip.y) / 2
    torso_height = hip_y - shoulder_y

    if torso_height <= 0:
        return 0

    face_y = (nose.y * 0.7) + (eye_y * 0.3)
    head_clearance = (shoulder_y - face_y) / torso_height
    head_down_score = 1 - head_clearance

    return head_down_score

def is_head_down(landmarks, threshold=0.35):
    head_down_score = get_head_down_score(landmarks)
    return head_down_score > threshold, head_down_score

In [9]:
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
VisionRunningMode = mp.tasks.vision.RunningMode

PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions

pose_options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="pose_landmarker.task"),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1
)

pose_landmarker = PoseLandmarker.create_from_options(pose_options)

I0000 00:00:1780954119.356551   59367 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1780954119.364490   59395 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: AMD Radeon Graphics (radeonsi, rembrandt, LLVM 20.1.2, DRM 3.64, 6.17.0-29-generic)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1780954119.415946   59374 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780954119.437119   59372 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [32]:
video_path = "/home/joel/Documents/music_lesson_app/backend-fastapi/video/waltz_no2_practice.mp4" 

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

head_down_events = []

frame_index = 0

while cap.isOpened():
    ret, frame = cap.read()

    if not ret or frame is None:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    timestamp_ms = int((frame_index / fps) * 1000)

    result = pose_landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    time_seconds = frame_index / fps

    if result.pose_landmarks:
        landmarks = result.pose_landmarks[0]

        head_down, head_down_score = is_head_down(
            landmarks,
            threshold=0.40
        
        )

        if head_down:
            head_down_events.append({
                "time": time_seconds,
                "score": head_down_score
            })

    frame_index += 1

cap.release()

print(f"Head down frames detected: {len(head_down_events)}")

for event in head_down_events[:10]:
    print(
        f"Head down around {format_time(event['time'])}, "
        f"score={event['score']:.3f}"
    )

Head down frames detected: 0


In [21]:
head_down_times = [event["time"] for event in head_down_events]

segments = []

if head_down_times:
    start = head_down_times[0]
    prev = head_down_times[0]

    for t in head_down_times[1:]:
        if t - prev > 0.5:
            segments.append((start, prev))
            start = t
        prev = t

    segments.append((start, prev))

for start, end in segments:
    duration = end - start
    if duration >= 1.0:
        print(
            f"Head tilted down from "
            f"{format_time(start)} to {format_time(end)} "
            f"({duration:.1f}s)"
        )

In [22]:
if result.pose_landmarks:
    landmarks = result.pose_landmarks[0]

    right_shoulder = landmarks[12]
    right_elbow = landmarks[14]
    right_wrist = landmarks[16]

    print(right_wrist.x, right_wrist.y, right_wrist.z)

In [36]:
import cv2
import mediapipe as mp
import numpy as np

def torso_center(landmarks):
    left_shoulder = landmarks[11]
    right_shoulder = landmarks[12]
    left_hip = landmarks[23]
    right_hip = landmarks[24]

    shoulder_center = np.array([
        (left_shoulder.x + right_shoulder.x) / 2,
        (left_shoulder.y + right_shoulder.y) / 2
    ])

    hip_center = np.array([
        (left_hip.x + right_hip.x) / 2,
        (left_hip.y + right_hip.y) / 2
    ])

    return shoulder_center, hip_center

def calculate_frame_metrics(landmarks):
    shoulder_center, hip_center = torso_center(landmarks)

    head_down_score = get_head_down_score(landmarks)
    head_down = head_down_score > 0.35

    torso_vector = shoulder_center - hip_center
    torso_lean = abs(torso_vector[0])

    posture_value = np.array([
        shoulder_center[0],
        shoulder_center[1],
        hip_center[0],
        hip_center[1],
        torso_lean
    ])

    return head_down, head_down_score, torso_lean, posture_value

In [37]:
video_path = "/home/joel/Documents/music_lesson_app/backend-fastapi/video/waltz_no2_practice.mp4" 

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

total_pose_frames = 0
looking_down_frames = 0
torso_lean_values = []
posture_values = []
head_down_scores = []

frame_index = 0

while cap.isOpened():
    ret, frame = cap.read()

    if not ret or frame is None:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    timestamp_ms = int((frame_index / fps) * 1000)

    result = pose_landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )

    if result.pose_landmarks:
        landmarks = result.pose_landmarks[0]

        head_down, head_down_score, torso_lean, posture_value = calculate_frame_metrics(landmarks)

        total_pose_frames += 1

        if head_down:
            looking_down_frames += 1

        torso_lean_values.append(torso_lean)
        posture_values.append(posture_value)
        head_down_scores.append(head_down_score)

    frame_index += 1

cap.release()

In [38]:
HEAD_DOWN_THRESHOLD = 0.35
head_down_scores_array = np.array(head_down_scores)

looking_down_percent = (
    np.mean(head_down_scores_array > HEAD_DOWN_THRESHOLD) * 100
    if len(head_down_scores_array) > 0 else 0
)

print("Head-down score summary")
if len(head_down_scores_array) > 0:
    print("min:", round(float(np.min(head_down_scores_array)), 3))
    print("avg:", round(float(np.mean(head_down_scores_array)), 3))
    print("max:", round(float(np.max(head_down_scores_array)), 3))

print("\nThreshold test")
for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45]:
    percent = (
        np.mean(head_down_scores_array > threshold) * 100
        if len(head_down_scores_array) > 0 else 0
    )
    print(f"{threshold:.2f}: {percent:.1f}% looking down")


average_torso_lean = (
    float(np.mean(torso_lean_values))
    if len(torso_lean_values) > 0 else 0
)

posture_change_count = 0
posture_change_threshold = 0.08

for i in range(1, len(posture_values)):
    change = np.linalg.norm(posture_values[i] - posture_values[i - 1])

    if change > posture_change_threshold:
        posture_change_count += 1

video_posture_metrics = {
    "looking_down_percent": round(looking_down_percent),
    "posture_change_count": posture_change_count,
    "average_torso_lean": round(average_torso_lean, 3)
}

video_posture_metrics

Head-down score summary

Threshold test
0.20: 0.0% looking down
0.25: 0.0% looking down
0.30: 0.0% looking down
0.35: 0.0% looking down
0.40: 0.0% looking down
0.45: 0.0% looking down


{'looking_down_percent': 0, 'posture_change_count': 0, 'average_torso_lean': 0}

In [25]:
from IPython.display import Video

video_path = "/home/joel/Documents/music_lesson_app/backend-fastapi/video/waltz_no2_practice.mp4" 

Video(video_path, embed=True)
Video(video_path, width=720, height=405, embed=False)
